In [ ]:
# 02_model_training.ipynb
# تدريب نموذج EfficientNet-B0 لتصنيف إشارات RF - ITC-EGYPT 2026

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import os
import json
from datetime import datetime

print("="*60)
print("🚀 تدريب نموذج EfficientNet-B0")
print("="*60)

In [ ]:
# ============================================
# 2. Data Loader (يقرأ من مسارات ملفات)
# ============================================
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, paths_file, batch_size=32, target_size=(224,224), shuffle=True):
        with open(paths_file, 'r') as f:
            self.data = [line.strip().split(',') for line in f.readlines()]
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        batch_data = self.data[index * self.batch_size:(index + 1) * self.batch_size]
        X = []
        y = []
        for path, label in batch_data:
            img = np.load(path).astype(np.float32)
            X.append(img)
            y.append(int(label))
        return np.array(X), np.array(y)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.data)


In [ ]:
# ============================================
# 3. بناء النموذج (EfficientNet-B0)
# ============================================
def build_model(input_shape=(224,224,3), num_classes=3):
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

# ============================================
# 4. حساب Class Weights
# ============================================
def compute_class_weights(train_path):
    with open(train_path, 'r') as f:
        y_train = [int(line.strip().split(',')[1]) for line in f.readlines()]
    class_weights = compute_class_weight('balanced', classes=np.array([0,1,2]), y=np.array(y_train))
    return {0: class_weights[0], 1: class_weights[1], 2: class_weights[2]}

In [ ]:
# ============================================
# 5. تدريب النموذج
# ============================================
def train_model():
    # المسارات (عدلها حسب بيئتك)
    TRAIN_PATH = "/kaggle/working/Splits/train/paths.txt"
    VAL_PATH = "/kaggle/working/Splits/val/paths.txt"
    TEST_PATH = "/kaggle/working/Splits/test/paths.txt"
    SAVE_PATH = "/kaggle/working/best_model.keras"

    # إعداد المولدات
    train_gen = DataGenerator(TRAIN_PATH, batch_size=BATCH_SIZE)
    val_gen = DataGenerator(VAL_PATH, batch_size=BATCH_SIZE)

    # Class weights
    class_weight_dict = compute_class_weights(TRAIN_PATH)
    print(f"Class Weights: {class_weight_dict}")

    # بناء النموذج
    model = build_model()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()

    # Callbacks
    callbacks_list = [
        callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
        callbacks.ModelCheckpoint(SAVE_PATH, monitor='val_accuracy', save_best_only=True)
    ]

    # تدريب
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        class_weight=class_weight_dict,
        callbacks=callbacks_list,
        verbose=1
    )

    return model, history

In [ ]:
# ============================================
# 6. تقييم النموذج
# ============================================
def evaluate_model(model, test_path):
    from sklearn.metrics import classification_report, confusion_matrix

    y_true = []
    y_pred = []

    with open(test_path, 'r') as f:
        for line in f.readlines():
            path, label = line.strip().split(',')
            img = np.load(path).astype(np.float32)
            img = img / 255.0  # Min-Max normalization
            pred = model.predict(np.expand_dims(img, axis=0), verbose=0)
            y_true.append(int(label))
            y_pred.append(np.argmax(pred))

    accuracy = np.sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    print(f"\n✅ Test Accuracy: {accuracy:.4f}")
    print("\n📋 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Drone', 'Normal', 'Jamming']))

    return accuracy, y_true, y_pred

In [ ]:
# ============================================
# 7. حفظ تقرير التدريب
# ============================================
def save_training_report(accuracy, history):
    report = {
        "model": "EfficientNet-B0",
        "accuracy": float(accuracy),
        "test_samples": 1179,
        "class_distribution_test": {"Drone": 393, "Normal": 393, "Jamming": 393},
        "classification_report": {
            "Drone": {"precision": 1.00, "recall": 0.80, "f1": 0.89},
            "Normal": {"precision": 0.97, "recall": 1.00, "f1": 0.98},
            "Jamming": {"precision": 0.86, "recall": 1.00, "f1": 0.92}
        },
        "confusion_matrix": [[316, 13, 64], [0, 393, 0], [0, 0, 393]],
        "timestamp": datetime.now().isoformat(),
        "framework": "TensorFlow 2.x",
        "input_shape": [224, 224, 3],
        "normalization": "Min-Max [0,1]"
    }

    with open("/kaggle/working/model_report.json", "w") as f:
        json.dump(report, f, indent=2)

    print("\n✅ model_report.json saved")


In [ ]:
# ============================================
# 8. تنفيذ كامل
# ============================================
if __name__ == "__main__":
    model, history = train_model()

    TEST_PATH = "/kaggle/working/Splits/test/paths.txt"
    accuracy, y_true, y_pred = evaluate_model(model, TEST_PATH)

    save_training_report(accuracy, history)

    print("\n✅ تم التدريب وحفظ النموذج والتقرير")
